In [ ]:
# Table discovery for bulk onboarding (run by the lakefed_ingest_discover job; usually started from
# the operations app). Lists every user table of each requested source database with recommended
# control-row settings (discovery.py) and writes them to Lakebase for review. Catalog metadata
# only: no table data is read. Runs on the same compute and network path as ingestion.
#
# spec (JSON): {"src_type": "sqlserver", "requested_by": "...",
#               a UC connection: "src_connection": "...", read through remote_query with no secret,
#               or a host:       "src_host": "...", "src_port": 1433 | null, "secret_key": "..." | "secret_key_template": "creds-{database}",
#               "databases": ["db1", "db2"] | "*" (every database on the server),
#               "kind": "list" to only list the server's databases (Test connection), with status 'listed',
#               "pattern": "lfc_eng_%" (a LIKE filter for the listing),
#               "azure_credential": "...", "azure_resource_id": "/subscriptions/.../Microsoft.Sql/servers/..." to list an
#               Azure SQL server's databases through Azure instead of master (a UC service credential with Reader)}
import json, os, sys

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import ct_engine, discovery
from state_store import StateStore

def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)

discovery_id = widget("discovery_id")
spec = json.loads(widget("spec", "{}"))
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
discover_workers = int(widget("discover_workers", str(discovery.DEFAULT_WORKERS)))
max_queries_per_source = int(widget("max_queries_per_source", "0"))
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = widget("secret_scope", "lakefed_ingest")
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = widget("trust_server_certificate", "true")
problems = discovery.spec_errors(spec) + ([] if discovery_id else ["discovery_id is required"])
assert not problems, "; ".join(problems)
store = StateStore.connect_lakebase(db_instance, db_name)
# A login typed into the operations app goes only to the host of the connection it was typed for
# (ct_engine.typed_login_refusal), whatever a control row or a discovery spec names.
_login_scope = os.environ["LAKEFED_SQL_SECRET_SCOPE"]
ct_engine.guard_typed_logins(_login_scope, store.typed_logins(_login_scope))

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

TABLE_COLS = ["discovery_id", "src_database", "src_schema", "src_table", "row_count", "size_mb", "ct_enabled",
              "primary_key", "select_list", "excluded_columns", "rowversion_column", "identity_column",
              "load_type", "cursor_column", "partition_candidate", "notes"]
# The one Lakebase connection serves every worker and autocommits, so a database's table rows are written in a
# transaction only while this lock is held: no other worker's statement can land inside it.
lakebase_lock = Lock()
TABLE_BATCH = 1000     # rows to an insert statement, as the app's CONTROL_BATCH writes do

def start(db, status="running"):
    with lakebase_lock, store.connection.cursor() as cur:
        cur.execute(
            "insert into lakefed_ingest.discovery (discovery_id, src_type, src_connection, src_host, src_port, src_database, "
            "secret_key, requested_by, status, finished_at, listing) values (%s, %s, %s, %s, %s, %s, %s, %s, %s, "
            "case when %s = 'running' then null else now() end, %s::jsonb) on conflict (discovery_id, src_database) do update set "
            "status = excluded.status, error = null, started_at = now(), finished_at = excluded.finished_at, "
            "listing = excluded.listing",
            (discovery_id, spec.get("src_type") or "sqlserver", spec.get("src_connection"), spec.get("src_host"),
             spec.get("src_port"), db, None if spec.get("src_connection") else discovery.secret_key_for(spec, db),
             spec.get("requested_by"), status, status, json.dumps(discovery.listing_of(spec))))

def finish(db, status, tables=None, error=None):
    with lakebase_lock, store.connection.cursor() as cur:
        cur.execute("update lakefed_ingest.discovery set status = %s, tables = %s, error = %s, finished_at = now() "
                    "where discovery_id = %s and src_database = %s", (status, tables, error, discovery_id, db))

def write_tables(db, rows):
    """Replace one database's discovered tables in a single transaction, TABLE_BATCH rows to a statement."""
    one_row = f"({', '.join(['%s'] * len(TABLE_COLS))})"
    with lakebase_lock, store.connection.transaction(), store.connection.cursor() as cur:
        cur.execute("delete from lakefed_ingest.discovered_table where discovery_id = %s and src_database = %s",
                    (discovery_id, db))
        for at in range(0, len(rows), TABLE_BATCH):
            chunk = rows[at:at + TABLE_BATCH]
            cur.execute(f"insert into lakefed_ingest.discovered_table ({', '.join(TABLE_COLS)}) "
                        f"values {', '.join([one_row] * len(chunk))}", tuple(v for row in chunk for v in row))

def one(db):
    cfg = discovery.cfg_for(spec, db)
    try:
        start(db)
        found = discovery.discover_database(cfg)
        rows = [tuple({**t, "discovery_id": discovery_id, "src_database": db}.get(c) for c in TABLE_COLS) for t in found]
        write_tables(db, rows)
        finish(db, "ok", tables=len(rows))
        return db, len(rows), None
    except Exception as e:          # a source error or a Lakebase write: the database is failed, not left running
        try:
            finish(db, "failed", error=str(e)[:2000])
        except Exception:
            pass                    # the connection is gone: the finally below marks it on a new one
        return db, None, str(e)[:300]

# Test connection lists the server's databases and stops; "*" discovers every one of them. A connection that fails
# is recorded on the server's own row (master), so the app can say why.
databases = spec.get("databases")
if discovery.needs_listing(spec):
    try:
        credential = (dbutils.credentials.getServiceCredentialsProvider(spec["azure_credential"])
                      if discovery.lists_through_azure(spec) else None)
        listed = discovery.list_databases(spec, credential=credential)
    except Exception as e:
        start(discovery.SERVER_DATABASE)
        finish(discovery.SERVER_DATABASE, "failed", error=str(e)[:2000])
        store.close()
        raise
    if spec.get("kind") == "list":
        for db in listed:
            start(db, status="listed")
        store.close()
        dbutils.notebook.exit(json.dumps({"discovery_id": discovery_id, "databases": len(listed), "listed": True}))
    databases = listed
workers = discovery.workers_for(discover_workers, len(databases), max_queries_per_source)
try:
    with ThreadPoolExecutor(max_workers=workers) as ex:
        results = list(ex.map(one, databases))
finally:
    # Whatever stopped this run (an exception, a lost connection), no database of it stays 'running'.
    closer = StateStore.connect_lakebase(db_instance, db_name)
    with closer.connection.cursor() as cur:
        cur.execute("update lakefed_ingest.discovery set status = 'failed', finished_at = now(), "
                    "error = coalesce(error, 'the discovery job stopped before finishing this database') "
                    "where discovery_id = %s and status = 'running'", (discovery_id,))
    closer.close()
for db, n, err in results:
    print(f"{db}: {n} tables" if err is None else f"{db}: FAILED {err}")
failed = [db for db, _, err in results if err]
store.close()
summary = {"discovery_id": discovery_id, "databases": len(databases), "workers": workers, "failed": failed,
           "tables": sum(n or 0 for _, n, _ in results)}
dbutils.notebook.exit(json.dumps(summary))